In [3]:
!pip install pandas nltk
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Isha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Isha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load dataset

Read the 200‑email triage dataset from the data folder into a pandas dataframe.

In [4]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


### Clean email body text

Convert the body to lowercase and remove numbers/punctuation so rules work on simple tokens.

In [5]:
df['clean_text'] = (
    df['body']
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

df[['body', 'clean_text']].head()


,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr is due on please pay to ...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


### Removing Stopwords and Extracting Useful Keywords

In [6]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Isha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
1,your invoice of inr is due on please pay to ...,"[invoice, inr, due, please, pay, avoid, late, ..."
2,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
3,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."
4,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."


### Rule-based triage function

Classify each email into respond_or_act, notify_human, or ignore using keyword rules.

In [11]:
def triage_rule(text):
    t = str(text)
    # simple rules
    if any(k in t for k in ["invoice", "payment due", "due on", "overdue"]):
        return "respond_or_act"
    if any(k in t for k in ["reset password", "password reset", "security issue"]):
        return "notify_human"
    if any(k in t for k in ["promotion", "sale", "unsubscribe", "offer"]):
        return "ignore"
    # default
    return "respond_or_act"

In [ ]:
# Apply to the dataframe
df["triage"] = df["clean_text"].apply(triage_rule)
df[["clean_text", "triage"]].head()

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,respond_or_act
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


In [13]:
df["triage"].value_counts()

triage
respond_or_act    181
ignore             19
Name: count, dtype: int64

In [14]:
out_name = "../data/milestone1_isha-bhole.csv"
df.to_csv(out_name, index=False)
print("Saved:", out_name)

Saved: ../data/milestone1_isha-bhole.csv
